<a href="https://colab.research.google.com/github/ArtIC-TITECH/b3-proj-2026/blob/main/theme_G/theme_G.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# テーマG：Looped LLMの再帰回数と精度・収束性の関係

## 目的

Ouro-1.4Bの再帰回数を1〜16回に変え、計算量を増やすことで予測性能が改善するか、また出力分布と潜在表現が固定点へ近づくかを評価する。

- **独立変数**：再帰回数
- **統制変数**：16-bit重み、評価データ、系列長、復号方法
- **性能指標**：Perplexity、next-token accuracy
- **収束指標**：隣接step間のJS divergence、予測token変更率、hidden state相対変化
- **振動の補助指標**：2 step前の予測へ戻る割合
- **計算コスト**：step 1を1とした相対計算量

この実験では、次の2点を区別する。

1. **性能最適点**：Perplexityやaccuracyが最良になる再帰回数
2. **状態安定点**：隣接step間の出力・hidden state変化が十分小さくなる再帰回数

有限個の入力・有限回の観測で数学的な収束を証明することはできない。ここでは「収束・発散・飽和・振動のどの傾向と整合するか」を実験的に議論する。


## 環境セットアップ

Colabで「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選ぶ。次のセルでパッケージが更新された場合だけ、セッションを再起動して先頭から実行する。


In [ ]:
# Ouroの固定revisionに合わせ、Transformers周辺だけ互換版へ揃えます。
# torch / CUDAはColab標準環境をそのまま使います。
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ランタイムのタイプをGPU（T4）に変更してください。")

TARGETS = {
    "transformers": "4.55.0",
    "tokenizers": "0.21.4",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.0",
    "safetensors": "0.6.2",
    "Pillow": "11.3.0",
}

needs_install = []
for pkg, wanted in TARGETS.items():
    try:
        current = version(pkg)
    except PackageNotFoundError:
        current = None
    if current != wanted:
        needs_install.append(f"{pkg}=={wanted}")

RESTART_REQUIRED = bool(needs_install)
if needs_install:
    print("互換パッケージをインストールします:", needs_install)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--upgrade", "--force-reinstall",
            "--no-deps", "--only-binary=:all:", *needs_install,
        ],
        check=True,
    )
    print("\nインストール完了。『ランタイム』→『セッションを再起動』を1回実行してください。")
    print("再起動後は、このNotebookを先頭から実行してください。")
else:
    print("環境OK:", torch.__version__, torch.cuda.get_device_name(0))


## モジュールの読み込み・GPU確認


In [ ]:
import gc
import math
import time
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw

if globals().get("RESTART_REQUIRED", False):
    raise RuntimeError("パッケージ更新後です。ランタイムを再起動して先頭から実行してください。")

required_versions = {
    "transformers": "4.55.0",
    "tokenizers": "0.21.4",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.0",
    "safetensors": "0.6.2",
}
for pkg, wanted in required_versions.items():
    try:
        current = version(pkg)
    except PackageNotFoundError:
        current = None
    if current != wanted:
        raise RuntimeError(f"{pkg}={current!r} です。環境セットアップを再実行してください。")

ImageDraw.Draw(Image.new("RGB", (8, 8))).point((0, 0))

from datasets import load_dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

device = "cuda:0"
capability = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if capability[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16
BASELINE_LABEL = "BF16" if dtype == torch.bfloat16 else "FP16"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "Transformers:", version("transformers"))
print("Model dtype:", BASELINE_LABEL)


## 実験条件

実験前に次を予想する。

- 性能が最大になる再帰回数はいくつか。
- 16回まで再帰すると、出力分布とhidden stateの変化は0へ近づくか。
- 性能最適点と状態安定点は一致するか。

再帰回数にほぼ比例して計算量が増えるため、精度だけでなく計算コストも考慮する。


In [ ]:
MODEL_ID = "ByteDance/Ouro-1.4B"
MODEL_REVISION = "574fa66cb8bf5abdc979642d01cf2b79b16bfab1"

# 独立変数：再帰回数。1〜16を連続して測る。
RECURRENT_STEPS = list(range(1, 17))
MAX_RECURRENT_STEPS = max(RECURRENT_STEPS)
DEFAULT_RECURRENT_STEPS = 4

# 8回版より評価tokenを増やし、細かな差を安定して比較する。
SEQ_LEN = 256
NUM_SEQUENCES = 32
EVAL_BATCH_SIZE = 1

# JS divergenceは語彙全体を使うため、token位置を間引いて測る。
JS_POSITION_STRIDE = 8
JS_CHUNK_POSITIONS = 8

print("Recurrent sweep:", RECURRENT_STEPS)
print("Default recurrent steps:", DEFAULT_RECURRENT_STEPS)
print("Evaluation predictions:", SEQ_LEN * NUM_SEQUENCES)
print("JS probe positions/sequence:", math.ceil(SEQ_LEN / JS_POSITION_STRIDE))


## Ouro-1.4Bの読み込みと中間状態の確認


In [ ]:
globals().pop("model", None)
gc.collect()
torch.cuda.empty_cache()

config = AutoConfig.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
config.total_ut_steps = MAX_RECURRENT_STEPS
config.use_cache = False
config.early_exit_threshold = None
config.early_exit_step = None

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    config=config,
    trust_remote_code=True,
    torch_dtype=dtype,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
).to(device)
model.eval()

# Ouroのbase modelにも最大再帰回数を同期する。
model.model.total_ut_steps = MAX_RECURRENT_STEPS
model.early_exit_threshold = None
model.early_exit_step = None

with torch.inference_mode():
    encoded = tokenizer("The sky is blue because", return_tensors="pt")
    base_out, hidden_states, gate_list = model.model(
        input_ids=encoded.input_ids.to(device),
        attention_mask=encoded.attention_mask.to(device),
        use_cache=False,
    )
    assert len(hidden_states) == MAX_RECURRENT_STEPS
    assert all(torch.isfinite(h).all() for h in hidden_states)
    print("Intermediate states:", len(hidden_states), "CUDA forward: OK")
    del encoded, base_out, hidden_states, gate_list
    torch.cuda.empty_cache()


## 評価データ：WikiText-2


In [ ]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test",
)
texts = [text for text in dataset["text"] if text.strip()]
all_text = "\n\n".join(texts)

tokens_per_sequence = SEQ_LEN + 1
needed_tokens = NUM_SEQUENCES * tokens_per_sequence
token_ids = tokenizer(
    all_text,
    return_tensors="pt",
    add_special_tokens=False,
    truncation=True,
    max_length=needed_tokens,
).input_ids[0]

if token_ids.numel() < needed_tokens:
    raise RuntimeError(f"評価用tokenが不足しています: {token_ids.numel()} < {needed_tokens}")

eval_tokens = token_ids[:needed_tokens].view(NUM_SEQUENCES, tokens_per_sequence)
print("eval_tokens:", tuple(eval_tokens.shape))


## 精度・収束・振動の評価

最大16回まで1度だけ再帰計算し、各stepのhidden stateへ同じ`lm_head`を適用する。各深さを別々にforwardするより条件を揃えて効率よく比較できる。

指標は次のように読む。

- **Perplexity**：低いほど次token予測が良い。
- **Next-token accuracy**：高いほど正解tokenを1位に選べている。
- **JS divergence**：隣接stepの出力確率分布の差。0に近いほど出力分布が安定している。
- **Top-1 flip rate**：隣接stepで予測tokenが変わった割合。低いほど離散的な予測が安定している。
- **Hidden relative change**：隣接stepのhidden state変化量。0に近いほど潜在表現が固定点へ近い。
- **Lag-2 return rate**：2 step前の予測へ戻った割合。高い場合は周期2振動の可能性がある。

収束指標が小さくなっても性能が改善するとは限らない。性能と状態変化を別々に確認すること。


In [ ]:
@torch.inference_mode()
def mean_js_divergence(logits_a, logits_b, stride=8, chunk_positions=8):
    """隣接stepの出力分布間の平均Jensen-Shannon divergence。"""
    a = logits_a[:, ::stride, :]
    b = logits_b[:, ::stride, :]
    total_js, total_positions = 0.0, 0

    for pos in range(0, a.shape[1], chunk_positions):
        aa = a[:, pos:pos + chunk_positions, :].float()
        bb = b[:, pos:pos + chunk_positions, :].float()
        log_p, log_q = F.log_softmax(aa, -1), F.log_softmax(bb, -1)
        p, q = log_p.exp(), log_q.exp()
        m = 0.5 * (p + q)
        log_m = torch.log(m.clamp_min(1e-12))
        js = 0.5 * (
            (p * (log_p - log_m)).sum(-1)
            + (q * (log_q - log_m)).sum(-1)
        )
        total_js += js.sum().item()
        total_positions += js.numel()

    return total_js, total_positions


@torch.inference_mode()
def evaluate_recurrence_sweep(model, eval_tokens, recurrent_steps, batch_size=1):
    """1回の最大深度forwardから、各再帰stepの精度と変化量を測る。"""
    max_steps = max(recurrent_steps)
    model.config.total_ut_steps = max_steps
    model.model.total_ut_steps = max_steps
    model.early_exit_threshold = None
    model.early_exit_step = None

    nll = {s: 0.0 for s in recurrent_steps}
    correct = {s: 0 for s in recurrent_steps}
    counts = {s: 0 for s in recurrent_steps}
    js_sum = {s: 0.0 for s in recurrent_steps if s > 1}
    js_count = {s: 0 for s in recurrent_steps if s > 1}
    flip_sum = {s: 0 for s in recurrent_steps if s > 1}
    flip_count = {s: 0 for s in recurrent_steps if s > 1}
    hidden_change_sum = {s: 0.0 for s in recurrent_steps if s > 1}
    hidden_count = {s: 0 for s in recurrent_steps if s > 1}
    lag2_sum = {s: 0 for s in recurrent_steps if s > 2}
    lag2_count = {s: 0 for s in recurrent_steps if s > 2}

    start_time = time.perf_counter()
    for start in range(0, len(eval_tokens), batch_size):
        batch = eval_tokens[start:start + batch_size].to(device)
        inputs, labels = batch[:, :-1], batch[:, 1:]
        base_out, hidden_states, gate_list = model.model(
            input_ids=inputs, use_cache=False
        )

        prev_logits = prev_hidden = prev_pred = pred_two_back = None
        for step in recurrent_steps:
            hidden = hidden_states[step - 1]
            logits = model.lm_head(hidden)
            if not torch.isfinite(logits).all():
                raise RuntimeError(f"step={step}: logitsにNaN/Infがあります")

            for pos in range(0, labels.shape[1], 32):
                scores = logits[:, pos:pos + 32, :].float()
                target = labels[:, pos:pos + 32]
                loss = F.cross_entropy(
                    scores.reshape(-1, scores.shape[-1]),
                    target.reshape(-1),
                    reduction="sum",
                )
                nll[step] += loss.item()
                correct[step] += (scores.argmax(-1) == target).sum().item()
                counts[step] += target.numel()

            pred = logits.argmax(-1)
            if prev_logits is not None:
                value, n_positions = mean_js_divergence(
                    prev_logits,
                    logits,
                    stride=JS_POSITION_STRIDE,
                    chunk_positions=JS_CHUNK_POSITIONS,
                )
                js_sum[step] += value
                js_count[step] += n_positions

                changed = pred != prev_pred
                flip_sum[step] += changed.sum().item()
                flip_count[step] += changed.numel()

                rel_change = torch.linalg.vector_norm(
                    hidden.float() - prev_hidden.float(), dim=-1
                ) / (torch.linalg.vector_norm(prev_hidden.float(), dim=-1) + 1e-8)
                hidden_change_sum[step] += rel_change.sum().item()
                hidden_count[step] += rel_change.numel()

            if pred_two_back is not None:
                returned = (pred == pred_two_back) & (pred != prev_pred)
                lag2_sum[step] += returned.sum().item()
                lag2_count[step] += returned.numel()

            prev_logits = logits
            prev_hidden = hidden
            pred_two_back = prev_pred
            prev_pred = pred

        del batch, inputs, labels, base_out, hidden_states, gate_list
        done = min(start + batch_size, len(eval_tokens))
        if done % 4 == 0 or done == len(eval_tokens):
            print(f"Evaluated {done}/{len(eval_tokens)}  {time.perf_counter()-start_time:.1f}s")

    rows = []
    for step in recurrent_steps:
        mean_nll = nll[step] / counts[step]
        rows.append({
            "recurrent_steps": step,
            "perplexity": math.exp(mean_nll) if mean_nll < 709 else float("inf"),
            "token_accuracy_pct": 100.0 * correct[step] / counts[step],
            "js_from_prev": js_sum[step] / js_count[step] if step > 1 else np.nan,
            "top1_flip_rate_pct": (
                100.0 * flip_sum[step] / flip_count[step] if step > 1 else np.nan
            ),
            "hidden_rel_change": (
                hidden_change_sum[step] / hidden_count[step] if step > 1 else np.nan
            ),
            "lag2_return_rate_pct": (
                100.0 * lag2_sum[step] / lag2_count[step] if step > 2 else np.nan
            ),
            "relative_compute": float(step),
        })
    return pd.DataFrame(rows)


## 再帰回数1〜16の比較


In [ ]:
results_df = evaluate_recurrence_sweep(
    model,
    eval_tokens,
    RECURRENT_STEPS,
    batch_size=EVAL_BATCH_SIZE,
)

default_row = results_df.loc[
    results_df["recurrent_steps"] == DEFAULT_RECURRENT_STEPS
].iloc[0]
best_ppl_row = results_df.loc[results_df["perplexity"].idxmin()]
best_acc_row = results_df.loc[results_df["token_accuracy_pct"].idxmax()]

# 基準値・最良値との差
results_df["ppl_ratio_vs_step4"] = (
    results_df["perplexity"] / default_row["perplexity"]
)
results_df["accuracy_delta_vs_step4_pp"] = (
    results_df["token_accuracy_pct"] - default_row["token_accuracy_pct"]
)
results_df["ppl_ratio_vs_best"] = (
    results_df["perplexity"] / best_ppl_row["perplexity"]
)
results_df["accuracy_delta_vs_best_pp"] = (
    results_df["token_accuracy_pct"] - best_acc_row["token_accuracy_pct"]
)

# 1 step追加したときの限界改善量。正なら改善、負なら悪化。
results_df["marginal_ppl_improvement"] = -results_df["perplexity"].diff()
results_df["marginal_accuracy_gain_pp"] = results_df["token_accuracy_pct"].diff()

# 少ない計算量でPPLを更新したstepを簡易Pareto frontierとする。
best_so_far = float("inf")
frontier = []
for ppl in results_df["perplexity"]:
    is_new_best = ppl < best_so_far
    frontier.append(is_new_best)
    best_so_far = min(best_so_far, ppl)
results_df["ppl_pareto_frontier"] = frontier

results_df.to_csv("theme_G_recurrence_results.csv", index=False)

display_columns = [
    "recurrent_steps",
    "perplexity",
    "token_accuracy_pct",
    "js_from_prev",
    "top1_flip_rate_pct",
    "hidden_rel_change",
    "lag2_return_rate_pct",
    "marginal_ppl_improvement",
    "marginal_accuracy_gain_pp",
]
results_df[display_columns].round(6)


In [ ]:
# 図1：性能の全体像と、step 2以降の拡大表示
after_step1 = results_df[results_df["recurrent_steps"] >= 2]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(results_df["recurrent_steps"], results_df["perplexity"], marker="o")
axes[0].axvline(DEFAULT_RECURRENT_STEPS, color="tab:red", linestyle="--", label="default=4")
axes[0].scatter(best_ppl_row["recurrent_steps"], best_ppl_row["perplexity"],
                s=110, marker="*", color="gold", edgecolor="black", zorder=5, label="best PPL")
axes[0].set_yscale("log")
axes[0].set_title("A. Perplexity — all steps")
axes[0].set_ylabel("Perplexity (log scale, lower is better)")
axes[0].legend()

axes[1].plot(after_step1["recurrent_steps"], after_step1["perplexity"], marker="o")
axes[1].axvline(DEFAULT_RECURRENT_STEPS, color="tab:red", linestyle="--")
axes[1].scatter(best_ppl_row["recurrent_steps"], best_ppl_row["perplexity"],
                s=110, marker="*", color="gold", edgecolor="black", zorder=5)
axes[1].set_title("B. Perplexity — zoom after step 1")
axes[1].set_ylabel("Perplexity")

axes[2].plot(results_df["recurrent_steps"], results_df["token_accuracy_pct"], marker="o")
axes[2].axvline(DEFAULT_RECURRENT_STEPS, color="tab:red", linestyle="--")
axes[2].scatter(best_acc_row["recurrent_steps"], best_acc_row["token_accuracy_pct"],
                s=110, marker="*", color="gold", edgecolor="black", zorder=5)
axes[2].set_title("C. Next-token accuracy")
axes[2].set_ylabel("Accuracy (%)")

for ax in axes:
    ax.set_xlabel("Recurrent steps")
    ax.set_xticks(range(1, MAX_RECURRENT_STEPS + 1))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# 図2：状態変化と振動。縦軸の意味が異なるため図を分ける。
conv_df = results_df.dropna(subset=["js_from_prev"])
osc_df = results_df.dropna(subset=["lag2_return_rate_pct"])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(conv_df["recurrent_steps"], conv_df["js_from_prev"], marker="o")
axes[0].axhline(0.01, color="gray", linestyle=":", label="reference=0.01")
axes[0].set_yscale("log")
axes[0].set_title("D. Output-distribution change")
axes[0].set_ylabel("JS divergence from previous step")
axes[0].legend()

axes[1].plot(conv_df["recurrent_steps"], conv_df["hidden_rel_change"], marker="o")
axes[1].axhline(0.10, color="gray", linestyle=":", label="reference=0.10")
axes[1].set_yscale("log")
axes[1].set_title("E. Hidden-state change")
axes[1].set_ylabel("Relative change from previous step")
axes[1].legend()

axes[2].plot(conv_df["recurrent_steps"], conv_df["top1_flip_rate_pct"],
             marker="o", label="top-1 flip")
axes[2].plot(osc_df["recurrent_steps"], osc_df["lag2_return_rate_pct"],
             marker="s", label="lag-2 return")
axes[2].axhline(5.0, color="gray", linestyle=":", label="reference=5%")
axes[2].set_title("F. Prediction stability / oscillation")
axes[2].set_ylabel("Token positions (%)")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("Recurrent steps")
    ax.set_xticks(range(2, MAX_RECURRENT_STEPS + 1, 2))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# 図3：1 loop追加する価値がどこまであるか
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(results_df["recurrent_steps"], results_df["marginal_ppl_improvement"])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("G. Marginal PPL improvement per added loop")
axes[0].set_ylabel("Previous PPL - current PPL")

axes[1].bar(results_df["recurrent_steps"], results_df["marginal_accuracy_gain_pp"])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("H. Marginal accuracy gain per added loop")
axes[1].set_ylabel("Accuracy gain (percentage points)")

for ax in axes:
    ax.set_xlabel("Recurrent steps")
    ax.set_xticks(range(1, MAX_RECURRENT_STEPS + 1))
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
def geometric_change_ratio(df, metric):
    """隣接step間変化量の幾何平均比。1未満なら全体として縮小。"""
    values = df[metric].dropna().clip(lower=1e-12).to_numpy(float)
    return float(np.exp(np.mean(np.log(values[1:] / values[:-1]))))


best_ppl_step = int(best_ppl_row["recurrent_steps"])
best_acc_step = int(best_acc_row["recurrent_steps"])

# 「ほぼ最良」：最良PPLの10%以内、かつ最良accuracyから1 point以内。
near_best = results_df[
    (results_df["ppl_ratio_vs_best"] <= 1.10)
    & (results_df["accuracy_delta_vs_best_pp"] >= -1.0)
]

# 参考閾値をすべて満たす最初のstepを状態安定点とする。
stable_candidates = results_df[
    (results_df["js_from_prev"] < 0.02)
    & (results_df["hidden_rel_change"] < 0.15)
    & (results_df["top1_flip_rate_pct"] < 10.0)
]
stable_step = (
    int(stable_candidates.iloc[0]["recurrent_steps"])
    if len(stable_candidates) else None
)

summary_df = pd.DataFrame([
    {"item": "Minimum PPL", "step": best_ppl_step,
     "value": float(best_ppl_row["perplexity"])},
    {"item": "Maximum accuracy", "step": best_acc_step,
     "value": float(best_acc_row["token_accuracy_pct"])},
    {"item": "First state-stable candidate", "step": stable_step,
     "value": np.nan},
    {"item": "JS geometric change ratio", "step": np.nan,
     "value": geometric_change_ratio(results_df, "js_from_prev")},
    {"item": "Hidden-change geometric ratio", "step": np.nan,
     "value": geometric_change_ratio(results_df, "hidden_rel_change")},
])

print("Performance-optimal step (PPL):", best_ppl_step)
print("Performance-optimal step (accuracy):", best_acc_step)
print("Near-best steps:", near_best["recurrent_steps"].tolist())
print("First state-stable candidate:", stable_step)
print("PPL Pareto-frontier steps:",
      results_df.loc[results_df["ppl_pareto_frontier"], "recurrent_steps"].tolist())

summary_df.round(6)


## 生成文の比較

step 1・4・8・16の生成を、同じプロンプトと同じgreedy decodingで比較する。

- step 1から4で、回答の形式や内容はどう変わるか。
- step 4から16で、自動指標が改善しなくても文章は変化するか。
- 説明問題と計算問題で傾向は同じか。

生成比較は自動評価を置き換えるものではなく、数値の変化が実際の文章にどう表れるかを観察するために行う。


In [ ]:
def set_recurrent_steps(model, steps):
    model.config.total_ut_steps = int(steps)
    model.model.total_ut_steps = int(steps)
    model.config.early_exit_threshold = None
    model.config.early_exit_step = None
    model.early_exit_threshold = None
    model.early_exit_step = None


@torch.inference_mode()
def generate_at_depth(prompt, steps, max_new_tokens=32):
    set_recurrent_steps(model, steps)
    encoded = tokenizer(prompt, return_tensors="pt")
    ids = encoded.input_ids.to(device)
    mask = encoded.attention_mask.to(device)
    start = ids.shape[1]

    for _ in range(max_new_tokens):
        out = model(
            input_ids=ids,
            attention_mask=mask,
            use_cache=False,
            logits_to_keep=1,
        )
        next_id = out.logits[:, -1, :].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
        mask = torch.cat([mask, torch.ones_like(next_id)], dim=1)
        if tokenizer.eos_token_id is not None and next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0, start:], skip_special_tokens=True)


PROMPTS = [
    "Explain why the sky looks blue in three sentences.",
    "If a car travels 150 km in 3 hours, what is its average speed? Explain briefly.",
]
GENERATION_STEPS = [1, 4, 8, 16]

generation_rows = []
for steps in GENERATION_STEPS:
    for prompt_id, prompt in enumerate(PROMPTS, 1):
        response = generate_at_depth(prompt, steps)
        generation_rows.append({
            "steps": steps,
            "prompt_id": prompt_id,
            "prompt": prompt,
            "response": response,
        })
        print(f"\n[step={steps}, prompt={prompt_id}]\n{response}")

generation_df = pd.DataFrame(generation_rows)
generation_df.to_csv("theme_G_generation_results.csv", index=False)
set_recurrent_steps(model, MAX_RECURRENT_STEPS)
generation_df


## 課題

- 再帰回数1〜16について、Perplexityとnext-token accuracyを比較し、性能最適点を求める。
- 全体図とstep 2以降の拡大図を使い、step 1の大きな差に隠れた変化も説明する。
- JS divergence、予測token変更率、hidden state相対変化から、状態安定点を求める。
- 性能最適点と状態安定点が一致するか確認し、一致しない場合はその意味を考察する。
- 1 loop追加したときの限界改善量から、どこで追加計算の効果が小さくなるか判断する。
- step 1・4・8・16の生成文を2種類のプロンプトで比較し、自動指標との対応を議論する。
- 評価系列数を変更する、安定判定の閾値を変更する、量子化モデルで同じ実験を行う、のいずれか1つ以上を追加実験する。

**提出物**：研究上の問い、予想、実験条件、性能最適点、状態安定点、結果表・グラフ、追加実験、考察、結論、限界をまとめる。「再帰回数を増やせば必ず改善する」「状態が収束すれば精度も上がる」と仮定せず、観測結果から結論を導くこと。
